In [9]:
import pandas as pd
data = pd.read_excel("Poland_6digit_FULL_Scores.xlsx")
data.head(3)

,Kod,Nazwa,Synteza,Task_Index,task_full_text_PL,task_full_text_ENG,Weaviate Status1,Weaviate Status2,score25,prediction_justification_gemni1.5,mean6d_pl,SD6d_pl,isco08_1d,description_1d,label1d,label6d,potential25,tasks_per_kod,task_color
0,111101,Parlamentarzysta,Czynnie uczestniczy w pracach Sejmu lub Senatu...,0,czynne uczestniczenie w pracach Sejmu lub Sena...,Active participation in the works of the Sejm ...,success,success,0.35,"The task of ""Active participation in the works...",0.372,0.080629,1,Managers,1 - Managers,111101 - Parlamentarzysta,Minimal Exposure,20,Low
1,111101,Parlamentarzysta,Czynnie uczestniczy w pracach Sejmu lub Senatu...,1,wyrażanie swojego stanowiska oraz zgłaszanie w...,Expressing one's position and submitting propo...,success,success,0.35,The task of expressing one's position and subm...,0.372,0.080629,1,Managers,1 - Managers,111101 - Parlamentarzysta,Minimal Exposure,20,Low
2,111101,Parlamentarzysta,Czynnie uczestniczy w pracach Sejmu lub Senatu...,2,zwracanie się do Prezydium Sejmu / Senatu lub ...,Addressing the Presidium of the Sejm / Senate ...,success,success,0.35,The task of addressing the Presidium of the Se...,0.372,0.080629,1,Managers,1 - Managers,111101 - Parlamentarzysta,Minimal Exposure,20,Low


In [ ]:
# Read mapping
mapping = pd.read_csv("ISCO_mapping.csv")

# Ensure all codes are strings before slicing or joining
data['Kod'] = data['Kod'].astype(str)
mapping['isco08_4d'] = mapping['isco08_4d'].astype(str)

# Create isco08_4d from Kod and perform the merge
data1 = (
    data[['Kod', 'Nazwa', 'Task_Index', 'task_full_text_ENG', 'potential25', 'score25', 'task_color']]
    .assign(isco08_4d = data['Kod'].str[:4])
    .merge(mapping, how='left', on='isco08_4d')
)

# Convert all necessary fields to string for safe concatenation
cols_to_string = ['isco08_1d', 'isco08_2d', 'isco08_3d', 'isco08_4d',
                  'description_1d', 'description_2d', 'description_3d', 'description_4d',
                  'Nazwa', 'score25']

for col in cols_to_string:
    data1[col] = data1[col].astype(str)

# Create combined labels
data1 = data1.assign(
    isco08_1d = data1['isco08_1d'] + " - " + data1['description_1d'],
    isco08_2d = data1['isco08_2d'] + " - " + data1['description_2d'],
    isco08_3d = data1['isco08_3d'] + " - " + data1['description_3d'],
    isco08_4d = data1['isco08_4d'] + " - " + data1['description_4d'],
    isco08_6d = data1['Kod'] + " - " + data1['Nazwa'],
    isco08_6d_task = "(" + data1['score25'] + ") - " + data1['task_full_text_ENG']
)

# Select final columns
data1 = data1[['isco08_1d', 'isco08_2d', 'isco08_3d', 'isco08_4d',
               'isco08_6d', 'isco08_6d_task', 'potential25', 'task_color']]

data1.head(3)

,isco08_1d,isco08_2d,isco08_3d,isco08_4d,isco08_6d,isco08_6d_task,potential25,task_color
0,1.0 - Managers,"11.0 - Chief executives, senior officials and ...",111.0 - Legislators and senior officials,1111 - Legislators,111101 - Parlamentarzysta,(0.35) - Active participation in the works of ...,Minimal Exposure,Low
1,1.0 - Managers,"11.0 - Chief executives, senior officials and ...",111.0 - Legislators and senior officials,1111 - Legislators,111101 - Parlamentarzysta,(0.35) - Expressing one's position and submitt...,Minimal Exposure,Low
2,1.0 - Managers,"11.0 - Chief executives, senior officials and ...",111.0 - Legislators and senior officials,1111 - Legislators,111101 - Parlamentarzysta,(0.35) - Addressing the Presidium of the Sejm ...,Minimal Exposure,Low


In [13]:
import pandas as pd
import json

def build_hierarchy(df):
    hierarchy = {"name": "ISCO-08", "children": []}

    # Group up to occupation level
    grouped = df.groupby(["isco08_1d", "isco08_2d", "isco08_3d", "isco08_4d", "isco08_6d"])

    for (isco08_1d, isco08_2d, isco08_3d, isco08_4d, isco08_6d), group in grouped:
        # Level 1
        level1 = next((x for x in hierarchy["children"] if x["name"] == isco08_1d), None)
        if not level1:
            level1 = {"name": isco08_1d, "children": []}
            hierarchy["children"].append(level1)

        # Level 2
        level2 = next((x for x in level1["children"] if x["name"] == isco08_2d), None)
        if not level2:
            level2 = {"name": isco08_2d, "children": []}
            level1["children"].append(level2)

        # Level 3
        level3 = next((x for x in level2["children"] if x["name"] == isco08_3d), None)
        if not level3:
            level3 = {"name": isco08_3d, "children": []}
            level2["children"].append(level3)

        # Level 4
        level4 = next((x for x in level3["children"] if x["name"] == isco08_4d), None)
        if not level4:
            level4 = {"name": isco08_4d, "children": []}
            level3["children"].append(level4)

        # Level 5 (isco08_6d → gets risk from potential25)
        risk_6d = group["potential25"].iloc[0]
        level5 = next((x for x in level4["children"] if x["name"] == isco08_6d), None)
        if not level5:
            level5 = {"name": isco08_6d, "risk": risk_6d, "children": []}
            level4["children"].append(level5)

        # Level 6 (isco08_6d_task → gets risk from task_color)
        for _, row in group.iterrows():
            level5["children"].append({
                "name": row["isco08_6d_task"],
                "risk": row["task_color"]
            })

    return hierarchy

# Build and export
hierarchy_data = build_hierarchy(data1)
with open("output_data.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_data, f, ensure_ascii=False, indent=2)

print("✅ JSON saved as output_data.json")


✅ JSON saved as output_data.json
